# Cerebra · TRIBE v2 neuro-engagement server (Colab)

Hosts `facebook/tribev2` as a public HTTP scoring endpoint for the Pika **neuro-eval** / **neuro-optimize** skills.

**Setup:** `Runtime → Change runtime type → T4 GPU`. A Hugging Face `HF_TOKEN` with accepted LLaMA 3.2 access is optional: when unavailable, the server uses TRIBE's ungated video pathway. Set `TRIBEV2_MODALITIES=video,audio` to add Wav2Vec-BERT. Run the cells top to bottom. **The install cell restarts the kernel on purpose** — when it reconnects, continue from the *Verify* cell (do not re-run install). The last cell prints the URL + key to paste into the skill.

## 1. Install packages
This cell installs everything, then **auto-restarts the kernel** so the pinned NumPy loads against a clean process (avoids the `numpy.ufunc ... __module__` ABI crash). The session will say *Reconnecting* — that's expected.

In [ ]:
# 1. Install TRIBE v2 + serving deps, then AUTO-RESTART the kernel.
#
#    Do NOT import numpy in this cell. Colab already has a different numpy loaded
#    in memory; importing the freshly-pinned 2.2.6 build in the SAME process makes
#    new C-extensions bind to the stale module and `import numpy` crashes inside
#    _override___module__ (the AttributeError: 'numpy.ufunc' ... '__module__').
#    The fix is: install -> restart the kernel -> import (next cell).
import importlib.metadata as metadata
import shutil, subprocess, sys

def pip(*pkgs, force=False):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"]
    if force: cmd.append("--force-reinstall")
    subprocess.run(cmd + list(pkgs), check=True)

# Idempotency guard: if a prior run already installed everything (e.g. you are
# re-running this cell AFTER the restart), skip install + restart so the notebook
# just continues. Safe to import here — on the very first run tribev2 is absent,
# so we fall through to install + restart.
try:
    import numpy, transformers, tribev2  # noqa
    _ready = (
        numpy.__version__ == "2.2.6"
        and transformers.__version__ == "5.12.1"
        and metadata.version("torch").startswith("2.6.0")
        and metadata.version("torchaudio").startswith("2.6.0")
        and shutil.which("uvx") is not None
    )
except Exception:
    _ready = False
if _ready:
    print("Already installed (pinned NumPy/Transformers + TRIBE v2 + uv). Skipping — continue to the next cell.")
    raise SystemExit  # ends this cell cleanly without touching the kernel

print("Installing NumPy 2.2.6 (TRIBE v2's pin) ..."); pip("numpy==2.2.6", force=True)
print("Installing pinned TRIBE v2 revision ...");     pip("git+https://github.com/facebookresearch/tribev2.git@38b3db073a7fa5bfbfc7fdd894b1a3536e4553e3")
# TRIBE v2's audio path imports transformers.Wav2Vec2BertModel; Colab's default
# transformers doesn't expose it. Pin the version the model is known to work with
# (matches a verified local run). Installed AFTER tribev2 so it wins.
print("Pinning transformers==5.12.1 (Wav2Vec2BertModel) ...")
# TRIBE already installed Transformers' dependency set. Use --no-deps here:
# a force reinstall with dependencies upgrades NumPy behind TRIBE's exact
# numpy==2.2.6 pin.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "--force-reinstall", "--no-deps", "transformers==5.12.1",
    ],
    check=True,
)
print("Installing uv + OpenCV + MNE + serving deps ..."); pip(
    "uv>=0.8", "opencv-python-headless>=4.10", "ultralytics>=8.3", "mne>=1.6",
    "fastapi>=0.110", "uvicorn[standard]>=0.29", "python-multipart>=0.0.9"
)
# Some broad downstream requirements allow a newer NumPy. Restore TRIBE's
# exact pin last, then restart before any compiled extension imports it.
print("Restoring TRIBE's NumPy 2.2.6 pin ..."); pip("numpy==2.2.6", force=True)
# TRIBE requires torch<2.7, so pip downgrades current Colab to torch 2.6/cu124.
# Colab's preinstalled torchaudio can remain at a newer ABI (for example
# 2.11/cu128), which crashes on import with `aoti_torch_abi_version`. Match it
# explicitly without allowing pip to replace the already-correct torch build.
print("Matching torchaudio to TRIBE's torch 2.6/cu124 ABI ...")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "--force-reinstall", "--no-deps",
        "--index-url", "https://download.pytorch.org/whl/cu124",
        "torchaudio==2.6.0+cu124",
    ],
    check=True,
)

print("\nInstall done — restarting the kernel so the new NumPy loads cleanly.")
print("When it reconnects, run the cells BELOW (do NOT re-run this one).")
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)


## 2. Verify (run after the restart)

In [ ]:
# 2. Verify — run this AFTER the kernel has restarted.
import shutil, subprocess
import cv2, numpy, torch, torchaudio, transformers, tribev2, ultralytics

assert torch.cuda.is_available(), (
    "CUDA is unavailable. In Colab choose Runtime → Change runtime type → T4 GPU, "
    "then reconnect and run this cell again."
)
assert numpy.__version__ == "2.2.6", f"Expected NumPy 2.2.6, got {numpy.__version__}"
assert torchaudio.__version__.startswith("2.6.0"), (
    f"Expected torchaudio 2.6.x for torch 2.6, got {torchaudio.__version__}"
)
assert shutil.which("ffmpeg"), "ffmpeg is missing"
assert shutil.which("uvx"), "uvx is missing; re-run the install cell once"
from transformers import Wav2Vec2BertModel  # noqa: F401

print("OK — NumPy", numpy.__version__,
      "| OpenCV", cv2.__version__,
      "| Ultralytics", ultralytics.__version__,
      "| Transformers", transformers.__version__,
      "| Torch", torch.__version__,
      "| Torchaudio", torchaudio.__version__,
      "| GPU", torch.cuda.get_device_name(0),
      "| tribev2 imported cleanly")


## 3. Write the scorer, OpenCV/YOLO features, ad-score formula, and API
These are the exact source files from the plugin's `server/` folder.

In [ ]:
%%writefile /content/engagement.py
"""TRIBE v2 -> engagement score.

Turns the raw cortical-response tensor predicted by `facebook/tribev2`
(`(T, 20484)` over the stacked fsaverage5 surface) into a compact, 0-100
"neuro-engagement" report: an overall score, four cognitively-labelled family
traces, and the peak moment.

This is a standalone port of the scoring path in Cerebra's `worker/app.py` --
the WebGL surface-mesh code is intentionally dropped; the plugin only needs the
number and the per-frame traces. Pure numpy, plus MNE once at startup to build
the Glasser/HCP-MMP atlas (cached to an .npz so later starts need no download).

Design notes:
- Engagement families are unions of named HCP-MMP (Glasser) parcels, chosen
  from the engagement-neuroscience literature, defined on TRIBE v2's *native*
  output space (no hand-placed spheres, no interpolation).
- Each timestep is standardized across the full cortical surface, then selected
  ROI activation is measured relative to the rest of the cortex. This avoids the
  former mathematical failure where temporal centering forced every clip's mean
  score back toward 50. All four families weight equally; `reliability` is an
  honest confidence note but never changes the score.
"""

from __future__ import annotations

import threading
from pathlib import Path

import numpy as np

# fsaverage5: 10,242 vertices per hemisphere = TRIBE v2's native resolution.
FS5_VERTICES_PER_HEMI = 10242

# Relative cortical scale: a family mean equal to the cortex mean -> 50, and
# +/-CORTICAL_Z_REF SD from the full-cortex mean -> 100/0.
CORTICAL_Z_REF = 2.5

# Four engagement families on the Glasser atlas. `rois` patterns support a
# trailing '*' (prefix match) or leading '*' (suffix match). `reliability`
# records how well TRIBE itself predicts that territory (paper Sec. 3.2-3.3):
# auditory/language are near the noise ceiling; association areas are well
# predicted; primary visual is weaker, so it is intentionally excluded.
ENGAGEMENT_FAMILIES = [
    {"key": "auditory_engagement", "name": "Auditory / speech-music", "short": "AUD",  "color": "#ffb13b",
     "reliability": "high",   "rois": ["A1", "MBelt", "LBelt", "PBelt", "A4", "A5", "STG*", "STS*"]},
    {"key": "language_message",    "name": "Language / message",      "short": "LANG", "color": "#ff5a7a",
     "reliability": "high",   "rois": ["44", "45", "47l", "IFS*", "IFJ*"]},
    {"key": "attention_salience",  "name": "Attention + salience",    "short": "ATTN", "color": "#9b8cff",
     "reliability": "medium", "rois": ["IPS*", "LIP*", "VIP", "FEF", "6a", "AVI", "MI", "FOP*", "a24pr", "p24pr", "PFm", "PGi", "PGs", "TPOJ*"]},
    {"key": "visual_motion",       "name": "Visual / motion",        "short": "VIS",  "color": "#3fd6c0",
     "reliability": "medium", "rois": ["MT", "MST", "V4t", "FST", "LO*", "V3CD"]},
]

_HCP_LOCK = threading.Lock()


def _build_hcp_labels(cache_dir: Path) -> dict:
    """Replicate `tribev2.utils.get_hcp_labels` WITHOUT its ~1.65 GB MNE *sample*
    dataset. We only need a subjects_dir containing `fsaverage`, so we use the
    small `fetch_fsaverage` (~tens of MB) + the HCP-MMP annotation, apply the
    same fsaverage->fsaverage5 downsample (keep vertices < 10242) and L/R stack.
    Result is cached to an .npz so later starts need no MNE.
    """
    cache_file = cache_dir / "hcp_fsaverage5_labels.npz"
    if cache_file.exists():
        data = np.load(cache_file, allow_pickle=False)
        return {k[4:]: data[k] for k in data.files}  # strip the "roi_" key prefix

    import mne

    fs5 = FS5_VERTICES_PER_HEMI
    subjects_dir = cache_dir / "mne_subjects"
    subjects_dir.mkdir(parents=True, exist_ok=True)
    mne.datasets.fetch_fsaverage(subjects_dir=str(subjects_dir), verbose=False)
    mne.datasets.fetch_hcp_mmp_parcellation(
        subjects_dir=str(subjects_dir), accept=True, verbose=False
    )
    labels = mne.read_labels_from_annot(
        "fsaverage", "HCPMMP1", hemi="both", subjects_dir=str(subjects_dir), verbose=False
    )

    out: dict[str, list] = {}
    for label in labels:
        name = label.name[2:].replace("_ROI", "")  # strip "L_"/"R_" prefix + "_ROI"
        if "-lh" in name:
            offset = 0
        elif "-rh" in name:
            offset = fs5
        else:
            continue
        bare = name.replace("-rh", "").replace("-lh", "")
        if not bare or "?" in bare:  # skip the medial-wall / unknown label
            continue
        verts = np.asarray(label.vertices)
        verts = verts[verts < fs5] + offset  # fsaverage meshes are nested: <fs5 == fsaverage5
        out.setdefault(bare, []).append(verts)
    merged = {k: np.concatenate(vs).astype(np.int64) for k, vs in out.items()}

    cache_dir.mkdir(parents=True, exist_ok=True)
    np.savez(cache_file, **{f"roi_{k}": v for k, v in merged.items()})
    return merged


def _match_rois(patterns, keys: list[str]) -> list[str]:
    """Resolve ROI name patterns (leading or trailing '*') to concrete Glasser
    parcel names present in the atlas."""
    out: list[str] = []
    for p in patterns:
        if p.endswith("*"):
            matched = [k for k in keys if k.startswith(p[:-1])]
        elif p.startswith("*"):
            matched = [k for k in keys if k.endswith(p[1:])]
        else:
            matched = [k for k in keys if k == p]
        if not matched:
            print(f"[scoring] warning: ROI pattern {p!r} matched no Glasser parcel")
        out.extend(matched)
    return sorted(set(out))


class EngagementScorer:
    """Holds the atlas->vertex mapping (built once) and scores prediction
    tensors. Construct one per process and reuse it."""

    def __init__(self, cache_dir: str | Path, tr_seconds: float):
        self.cache_dir = Path(cache_dir)
        self.tr = float(tr_seconds)
        self._families: list[dict] | None = None
        self._labels: dict | None = None

    def warmup(self) -> None:
        """Build the atlas eagerly (call once at startup, off the request path)."""
        self._family_rois()

    def _hcp_labels(self) -> dict:
        if self._labels is None:
            with _HCP_LOCK:
                if self._labels is None:
                    self._labels = _build_hcp_labels(self.cache_dir)
        return self._labels

    def _family_rois(self) -> list[dict]:
        """Per family: {parcel_name -> vertex indices}. Kept per-parcel so we can
        average within a parcel before averaging across parcels (parcel-balanced,
        so a large parcel doesn't dominate a family)."""
        if self._families is None:
            labels = self._hcp_labels()
            keys = list(labels.keys())
            families = []
            for fam in ENGAGEMENT_FAMILIES:
                names = _match_rois(fam["rois"], keys)
                families.append({n: np.asarray(labels[n], dtype=np.int64) for n in names})
            self._families = families
        return self._families

    def score(self, predictions: np.ndarray) -> dict:
        """Summarise predicted surface responses over the four engagement
        families into a 0-100 report. `predictions` is `(T, 20484)`."""
        pred = np.asarray(predictions, dtype=np.float64)
        if pred.ndim != 2:
            raise ValueError(
                f"TRIBE predictions must be a 2D (time, vertices) array; got {pred.shape}"
            )
        if pred.shape[1] != 2 * FS5_VERTICES_PER_HEMI:
            raise ValueError(
                "TRIBE predictions must use the fsaverage5 surface "
                f"({2 * FS5_VERTICES_PER_HEMI} vertices); got {pred.shape[1]}"
            )
        if pred.shape[0] == 0:
            raise ValueError("TRIBE returned zero prediction frames")
        if not np.isfinite(pred).all():
            raise ValueError("TRIBE predictions contain NaN or infinite values")
        n_frames = int(pred.shape[0])

        # Standardize across the cortex independently at each timestep. The
        # temporal average is therefore free to vary by clip; unlike temporal
        # per-vertex centering, this does not make the headline score identically 50.
        mu = pred.mean(axis=1, keepdims=True)
        sd = pred.std(axis=1, keepdims=True)
        sd[sd < 1e-6] = 1e-6
        z = (pred - mu) / sd

        families = self._family_rois()
        traces = np.zeros((len(ENGAGEMENT_FAMILIES), n_frames), dtype=np.float32)
        for i, parcels in enumerate(families):
            if not parcels:
                continue
            # Mean within each parcel, then mean across parcels (parcel-balanced).
            parcel_means = [z[:, idx].mean(axis=1) for idx in parcels.values()]
            fam_z = np.mean(np.stack(parcel_means, axis=0), axis=0)
            traces[i] = np.clip(
                50.0 * (1.0 + fam_z / CORTICAL_Z_REF),
                0,
                100,
            )

        regions = []
        cognitive_series = {}
        for i, fam in enumerate(ENGAGEMENT_FAMILIES):
            values = np.round(traces[i], 1).tolist()
            cognitive_series[fam["key"]] = values
            regions.append({
                "name": fam["name"],
                "short": fam["short"],
                "color": fam["color"],
                "reliability": fam["reliability"],
                "score": round(float(traces[i].mean()), 1) if n_frames else 0.0,
                "values": values,
            })

        # Overall engagement = equal-weighted mean of the four families.
        global_trace = traces.mean(axis=0)
        activation_score = round(float(np.mean([r["score"] for r in regions])), 1)
        regions.sort(key=lambda r: r["score"], reverse=True)

        peak_index = int(np.argmax(global_trace)) if n_frames else 0
        trough_index = int(np.argmin(global_trace)) if n_frames else 0
        peak_label = regions[0]["short"] if regions else "CORTEX"
        return {
            "duration": round(n_frames * self.tr, 2),
            "frames": n_frames,
            "tr": round(self.tr, 4),
            "source": "model",
            "activationScore": activation_score,
            "global": np.round(global_trace, 2).tolist(),
            "regions": regions,
            "cognitiveSeries": cognitive_series,
            "peak": {
                "time": round(peak_index * self.tr, 2),
                "label": peak_label,
                "value": round(float(global_trace[peak_index]), 2) if n_frames else 0.0,
            },
            "trough": {
                "time": round(trough_index * self.tr, 2),
                "value": round(float(global_trace[trough_index]), 2) if n_frames else 0.0,
            },
            "weakWindow": _weak_window(global_trace, self.tr),
        }


def _weak_window(global_trace: np.ndarray, tr: float, win: int = 3) -> dict:
    """Lowest-engagement contiguous window (the beat an optimizer should
    regenerate first). Returns the [start, end] time span and its mean score."""
    n = len(global_trace)
    if n == 0:
        return {"startTime": 0.0, "endTime": 0.0, "meanValue": 0.0}
    w = min(win, n)
    sums = np.convolve(global_trace, np.ones(w, dtype=np.float64), mode="valid")
    start = int(np.argmin(sums))
    return {
        "startTime": round(start * tr, 2),
        "endTime": round((start + w) * tr, 2),
        "meanValue": round(float(sums[start] / w), 2),
    }


In [ ]:
%%writefile /content/video_features.py
"""Deterministic short-video normalization and OpenCV feature extraction."""

from __future__ import annotations

import math
import shutil
import subprocess
from pathlib import Path

import cv2
import numpy as np

MAX_CLIP_SECONDS = 3.0
DEFAULT_YOLO_MODEL = "yolov8n.pt"


def normalize_short_video(
    source: str | Path,
    destination: str | Path,
    *,
    max_seconds: float = MAX_CLIP_SECONDS,
) -> dict:
    """Transcode to a browser/model-safe MP4 and trim to ``max_seconds``."""
    source, destination = Path(source), Path(destination)
    if not shutil.which("ffmpeg"):
        raise RuntimeError("ffmpeg is required to normalize uploaded videos")

    original = extract_opencv_features(source, max_seconds=None)
    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel",
        "error",
        "-y",
        "-i",
        str(source),
        "-t",
        f"{max_seconds:.3f}",
        "-map",
        "0:v:0",
        "-map",
        "0:a?",
        "-vf",
        "scale=trunc(iw/2)*2:trunc(ih/2)*2",
        "-c:v",
        "libx264",
        "-preset",
        "veryfast",
        "-crf",
        "20",
        "-pix_fmt",
        "yuv420p",
        "-c:a",
        "aac",
        "-b:a",
        "128k",
        "-movflags",
        "+faststart",
        str(destination),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
    if result.returncode != 0 or not destination.exists():
        raise RuntimeError(f"ffmpeg normalization failed: {result.stderr.strip()}")

    normalized = extract_opencv_features(destination, max_seconds=max_seconds)
    normalized["original_duration"] = original["duration"]
    normalized["trimmed"] = original["duration"] > max_seconds + 0.05
    return normalized


def extract_opencv_features(
    video_path: str | Path,
    *,
    max_seconds: float | None = MAX_CLIP_SECONDS,
    max_frames: int = 90,
) -> dict:
    """Read a short clip with OpenCV and return interpretable visual features."""
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise ValueError("OpenCV could not open the uploaded video")

    fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    if fps <= 0 or width <= 0 or height <= 0:
        capture.release()
        raise ValueError("Video metadata is invalid (fps or dimensions are zero)")

    metadata_duration = frame_count / fps if frame_count > 0 else 0.0
    frame_limit = max_frames
    if max_seconds is not None:
        frame_limit = min(frame_limit, max(1, int(math.ceil(fps * max_seconds))))

    brightness, contrast, saturation, sharpness = [], [], [], []
    motion, histogram_deltas = [], []
    previous_gray = previous_hist = None
    decoded = 0

    while decoded < frame_limit:
        ok, frame = capture.read()
        if not ok:
            break
        decoded += 1
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        brightness.append(float(gray.mean() / 255.0))
        contrast.append(float(gray.std() / 255.0))
        saturation.append(float(hsv[..., 1].mean() / 255.0))
        sharpness.append(float(cv2.Laplacian(gray, cv2.CV_64F).var()))

        if previous_gray is not None:
            motion.append(float(cv2.absdiff(gray, previous_gray).mean() / 255.0))
        hist = cv2.calcHist([gray], [0], None, [32], [0, 256])
        cv2.normalize(hist, hist)
        if previous_hist is not None:
            correlation = cv2.compareHist(previous_hist, hist, cv2.HISTCMP_CORREL)
            histogram_deltas.append(float(np.clip(1.0 - correlation, 0.0, 2.0) / 2.0))
        previous_gray, previous_hist = gray, hist

    capture.release()
    if decoded == 0:
        raise ValueError("OpenCV decoded zero frames")

    duration = metadata_duration or decoded / fps
    if max_seconds is not None:
        duration = min(duration, max_seconds)

    values = {
        "duration": round(float(duration), 3),
        "fps": round(fps, 3),
        "width": width,
        "height": height,
        "sampled_frames": decoded,
        "brightness": round(float(np.mean(brightness)), 4),
        "contrast": round(float(np.mean(contrast)), 4),
        "saturation": round(float(np.mean(saturation)), 4),
        "sharpness": round(float(np.mean(sharpness)), 2),
        "motion_energy": round(float(np.mean(motion)) if motion else 0.0, 4),
        "scene_change_rate": round(
            float(np.mean(np.asarray(histogram_deltas) > 0.22))
            if histogram_deltas
            else 0.0,
            4,
        ),
    }
    values["visual_score"] = _visual_score(values)
    return values


_YOLO_CACHE: dict[str, object] = {}


def extract_yolo_features(
    video_path: str | Path,
    *,
    model_path: str = DEFAULT_YOLO_MODEL,
    sample_frames: int = 8,
) -> dict:
    """Extract lightweight semantic/composition features with YOLO.

    The scorer degrades gracefully when Ultralytics or weights are unavailable;
    the weighted ad-score builder redistributes unavailable feature weights.
    """
    try:
        from ultralytics import YOLO
    except Exception as exc:
        return {
            "available": False,
            "reason": f"ultralytics unavailable: {type(exc).__name__}: {exc}",
        }

    try:
        model = _YOLO_CACHE.get(model_path)
        if model is None:
            model = YOLO(model_path)
            _YOLO_CACHE[model_path] = model

        capture = cv2.VideoCapture(str(video_path))
        total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
        if total <= 0 or width <= 0 or height <= 0:
            capture.release()
            raise ValueError("invalid video metadata for YOLO")

        indices = np.linspace(0, total - 1, min(sample_frames, total), dtype=int)
        frames = []
        for index in indices:
            capture.set(cv2.CAP_PROP_POS_FRAMES, int(index))
            ok, frame = capture.read()
            if ok:
                frames.append(frame)
        capture.release()
        if not frames:
            raise ValueError("YOLO sampling decoded zero frames")

        results = model.predict(frames, verbose=False, conf=0.25)
        focal_areas, confidences, object_counts, center_scores = [], [], [], []
        detected_frames = 0
        class_names: set[str] = set()
        for result in results:
            boxes = result.boxes
            count = len(boxes) if boxes is not None else 0
            object_counts.append(count)
            if not count:
                focal_areas.append(0.0)
                confidences.append(0.0)
                center_scores.append(0.0)
                continue
            detected_frames += 1
            xyxy = boxes.xyxy.detach().cpu().numpy()
            conf = boxes.conf.detach().cpu().numpy()
            cls = boxes.cls.detach().cpu().numpy().astype(int)
            frame_area = float(width * height)
            areas = (
                (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1]) / frame_area
            )
            best = int(np.argmax(areas * np.maximum(conf, 1e-6)))
            focal_areas.append(float(np.clip(areas[best], 0, 1)))
            confidences.append(float(conf[best]))
            cx = float((xyxy[best, 0] + xyxy[best, 2]) / 2 / width)
            cy = float((xyxy[best, 1] + xyxy[best, 3]) / 2 / height)
            distance = math.sqrt((cx - 0.5) ** 2 + (cy - 0.5) ** 2)
            center_scores.append(float(np.clip(1.0 - distance / 0.7071, 0, 1)))
            names = result.names
            class_names.update(str(names.get(int(c), c)) for c in cls)

        return {
            "available": True,
            "sampled_frames": len(frames),
            "detection_coverage": round(detected_frames / len(frames), 4),
            "mean_object_count": round(float(np.mean(object_counts)), 3),
            "focal_area_ratio": round(float(np.mean(focal_areas)), 4),
            "focal_confidence": round(float(np.mean(confidences)), 4),
            "focal_centering": round(float(np.mean(center_scores)), 4),
            "classes": sorted(class_names),
        }
    except Exception as exc:
        return {
            "available": False,
            "reason": f"YOLO inference failed: {type(exc).__name__}: {exc}",
        }


def _visual_score(features: dict) -> float:
    """A bounded production-quality prior; TRIBE remains the primary signal."""
    brightness = float(features["brightness"])
    contrast = float(features["contrast"])
    saturation = float(features["saturation"])
    sharpness = float(features["sharpness"])
    motion = float(features["motion_energy"])
    scene_rate = float(features["scene_change_rate"])

    exposure = np.clip(100.0 - abs(brightness - 0.5) * 180.0, 0, 100)
    contrast_score = np.clip(contrast / 0.24 * 100.0, 0, 100)
    color_score = np.clip(100.0 - abs(saturation - 0.35) * 150.0, 0, 100)
    sharpness_score = np.clip(sharpness / 850.0 * 100.0, 0, 100)
    # Three-second ads benefit from visible motion, but extreme frame-to-frame
    # churn is usually compression/noise rather than meaningful dynamism.
    motion_score = np.clip(100.0 - abs(motion - 0.08) / 0.08 * 100.0, 0, 100)
    cut_score = np.clip(100.0 - max(0.0, scene_rate - 0.34) * 150.0, 0, 100)

    score = (
        0.20 * exposure
        + 0.15 * contrast_score
        + 0.10 * color_score
        + 0.20 * sharpness_score
        + 0.25 * motion_score
        + 0.10 * cut_score
    )
    return round(float(np.clip(score, 0, 100)), 1)


In [ ]:
%%writefile /content/ad_score.py
"""Auditable linear ad-score over TRIBE, OpenCV, and YOLO features."""

from __future__ import annotations

from typing import Any

import numpy as np


FEATURE_WEIGHTS = {
    # TRIBE cortical activation features: 55%
    "tribe_attention": 0.18,
    "tribe_visual": 0.14,
    "tribe_language": 0.13,
    "tribe_auditory": 0.10,
    # OpenCV production features: 30%
    "opencv_exposure": 0.06,
    "opencv_contrast": 0.05,
    "opencv_sharpness": 0.07,
    "opencv_motion": 0.08,
    "opencv_pacing": 0.04,
    # YOLO semantic/composition features: 15%
    "yolo_detection_coverage": 0.04,
    "yolo_focal_size": 0.04,
    "yolo_focal_confidence": 0.03,
    "yolo_centering": 0.04,
}


def _clamp(value: float) -> float:
    return float(np.clip(value, 0.0, 100.0))


def _target_score(value: float, target: float, tolerance: float) -> float:
    return _clamp(100.0 - abs(value - target) / tolerance * 100.0)


def _tribe_region_scores(report: dict[str, Any]) -> dict[str, float]:
    by_short = {
        region["short"]: float(region["score"])
        for region in report.get("regions", [])
    }
    return {
        "tribe_attention": by_short.get("ATTN"),
        "tribe_visual": by_short.get("VIS"),
        "tribe_language": by_short.get("LANG"),
        "tribe_auditory": by_short.get("AUD"),
    }


def _optional_feature(
    source: str,
    raw_value: Any,
    score_fn,
    *,
    reason: str,
) -> dict[str, Any]:
    if raw_value is None:
        return {
            "source": source,
            "raw_value": None,
            "score": None,
            "unavailable_reason": reason,
        }
    return {
        "source": source,
        "raw_value": raw_value,
        "score": _clamp(float(score_fn(float(raw_value)))),
    }


def build_ad_score(
    tribe_report: dict[str, Any],
    opencv: dict[str, Any],
    yolo: dict[str, Any],
) -> dict[str, Any]:
    """Return raw values, normalized scores, effective weights, contributions."""
    values: dict[str, dict[str, Any]] = {}

    for key, score in _tribe_region_scores(tribe_report).items():
        values[key] = {
            "source": "tribev2",
            "raw_value": score,
            "score": None if score is None else _clamp(score),
        }

    values.update(
        {
            "opencv_exposure": _optional_feature(
                "opencv",
                opencv.get("brightness"),
                lambda value: _target_score(value, 0.5, 0.5),
                reason="brightness unavailable",
            ),
            "opencv_contrast": _optional_feature(
                "opencv",
                opencv.get("contrast"),
                lambda value: value / 0.24 * 100.0,
                reason="contrast unavailable",
            ),
            "opencv_sharpness": _optional_feature(
                "opencv",
                opencv.get("sharpness"),
                lambda value: value / 850.0 * 100.0,
                reason="sharpness unavailable",
            ),
            "opencv_motion": _optional_feature(
                "opencv",
                opencv.get("motion_energy"),
                lambda value: _target_score(value, 0.08, 0.08),
                reason="motion energy unavailable",
            ),
            "opencv_pacing": _optional_feature(
                "opencv",
                opencv.get("scene_change_rate"),
                lambda value: 100.0 - max(0.0, value - 0.34) * 150,
                reason="scene-change rate unavailable",
            ),
        }
    )

    if yolo.get("available"):
        values.update(
            {
                "yolo_detection_coverage": {
                    "source": "yolo",
                    "raw_value": yolo["detection_coverage"],
                    "score": _clamp(float(yolo["detection_coverage"]) * 100),
                },
                "yolo_focal_size": {
                    "source": "yolo",
                    "raw_value": yolo["focal_area_ratio"],
                    "score": _target_score(float(yolo["focal_area_ratio"]), 0.28, 0.28),
                },
                "yolo_focal_confidence": {
                    "source": "yolo",
                    "raw_value": yolo["focal_confidence"],
                    "score": _clamp(float(yolo["focal_confidence"]) * 100),
                },
                "yolo_centering": {
                    "source": "yolo",
                    "raw_value": yolo["focal_centering"],
                    "score": _clamp(float(yolo["focal_centering"]) * 100),
                },
            }
        )
    else:
        for key in (
            "yolo_detection_coverage",
            "yolo_focal_size",
            "yolo_focal_confidence",
            "yolo_centering",
        ):
            values[key] = {
                "source": "yolo",
                "raw_value": None,
                "score": None,
                "unavailable_reason": yolo.get("reason", "YOLO unavailable"),
            }

    available_weight = sum(
        FEATURE_WEIGHTS[key]
        for key, feature in values.items()
        if feature["score"] is not None
    )
    if available_weight <= 0:
        raise ValueError("No ad-score features are available")

    feature_rows = []
    source_totals: dict[str, float] = {}
    total = 0.0
    for key, base_weight in FEATURE_WEIGHTS.items():
        feature = values[key]
        effective_weight = (
            base_weight / available_weight if feature["score"] is not None else 0.0
        )
        contribution = (
            float(feature["score"]) * effective_weight
            if feature["score"] is not None
            else 0.0
        )
        source_totals[feature["source"]] = (
            source_totals.get(feature["source"], 0.0) + contribution
        )
        total += contribution
        feature_rows.append(
            {
                "name": key,
                **feature,
                "base_weight": base_weight,
                "effective_weight": round(effective_weight, 6),
                "contribution": round(contribution, 3),
            }
        )

    weakest = sorted(
        (row for row in feature_rows if row["score"] is not None),
        key=lambda row: row["score"],
    )[:3]
    strongest = sorted(
        (row for row in feature_rows if row["score"] is not None),
        key=lambda row: row["contribution"],
        reverse=True,
    )[:3]
    return {
        "adScore": round(total, 1),
        "formula": "sum(normalized_feature_score * effective_weight)",
        "weights_redistributed": available_weight < 0.999999,
        "features": feature_rows,
        "sourceContributions": {
            key: round(value, 3) for key, value in source_totals.items()
        },
        "weakestFeatures": [
            {"name": row["name"], "score": round(float(row["score"]), 1)}
            for row in weakest
        ],
        "strongestContributors": [
            {"name": row["name"], "contribution": row["contribution"]}
            for row in strongest
        ],
    }


In [ ]:
%%writefile /content/server.py
"""Cerebra neuro-engagement scoring server.

Wraps ``facebook/tribev2`` behind a small FastAPI API. The important reliability
property is that text is an optional enhancement, not a single point of failure:

* ``TRIBEV2_TEXT_MODE=auto`` (default) uses WhisperX + LLaMA when a token with
  LLaMA 3.2 access is available, then retries with the configured ungated
  modalities if either stage fails.
* ``TRIBEV2_TEXT_MODE=off`` needs no gated model. ``TRIBEV2_MODALITIES=video``
  is the universal T4 default; ``video,audio`` enables Wav2Vec-BERT too.
* ``TRIBEV2_TEXT_MODE=required`` fails clearly instead of silently degrading.

The public TRIBE checkpoint is still the same multimodal brain model in every
mode. Missing modality features are represented by zeros by TRIBE's own
inference pipeline.
"""

from __future__ import annotations

import gc
import os
import tempfile
import threading
import traceback
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Any

import numpy as np
import torch
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from starlette.concurrency import run_in_threadpool

from ad_score import FEATURE_WEIGHTS, build_ad_score
from engagement import EngagementScorer
from video_features import (
    MAX_CLIP_SECONDS,
    extract_yolo_features,
    normalize_short_video,
)

torch.set_float32_matmul_precision("high")
torch.backends.cudnn.allow_tf32 = True

MODEL_ID = os.getenv("TRIBEV2_MODEL_ID", "facebook/tribev2")
LLAMA_MODEL_ID = os.getenv("TRIBEV2_TEXT_MODEL_ID", "meta-llama/Llama-3.2-3B")
CACHE_DIR = Path(
    os.getenv(
        "TRIBEV2_CACHE_DIR",
        str(Path(__file__).resolve().parent / ".cache"),
    )
)
MAX_UPLOAD_BYTES = int(os.getenv("MAX_UPLOAD_BYTES", str(200_000_000)))
API_KEY = os.getenv("NEURO_API_KEY", "").strip()
TEXT_MODE = os.getenv("TRIBEV2_TEXT_MODE", "auto").strip().lower()
INFERENCE_MODALITIES = tuple(
    item.strip().lower()
    for item in os.getenv("TRIBEV2_MODALITIES", "video").split(",")
    if item.strip()
)
UPSTREAM_COMMIT = os.getenv("TRIBEV2_UPSTREAM_COMMIT", "unknown")
SERVER_VERSION = "0.3.0"
MAX_LOOP_ITERATIONS = 5
VIDEO_MODEL_PATH = os.getenv("TRIBEV2_VIDEO_MODEL_PATH", "").strip()
AUDIO_MODEL_PATH = os.getenv("TRIBEV2_AUDIO_MODEL_PATH", "").strip()
_BUNDLED_YOLO = Path(__file__).resolve().parent / "assets" / "yolov8n.pt"
YOLO_MODEL = os.getenv(
    "YOLO_MODEL",
    str(_BUNDLED_YOLO) if _BUNDLED_YOLO.exists() else "yolov8n.pt",
)

if TEXT_MODE not in {"auto", "off", "required"}:
    raise RuntimeError(
        "TRIBEV2_TEXT_MODE must be one of: auto, off, required "
        f"(got {TEXT_MODE!r})"
    )
if not INFERENCE_MODALITIES or INFERENCE_MODALITIES[0] != "video":
    raise RuntimeError(
        "TRIBEV2_MODALITIES must be `video` or `video,audio`; "
        f"got {','.join(INFERENCE_MODALITIES)!r}"
    )
if any(item not in {"video", "audio"} for item in INFERENCE_MODALITIES):
    raise RuntimeError(
        "TRIBEV2_MODALITIES supports only `video` and optional `audio`; "
        f"got {','.join(INFERENCE_MODALITIES)!r}"
    )

_state: dict[str, Any] = {
    "model": None,
    "scorer": None,
    "startup_error": None,
    "last_error": None,
    "text_enabled": False,
    "text_reason": "not checked",
}
_inference_lock = threading.Lock()


class PipelineFailure(RuntimeError):
    """A model-pipeline failure with enough context to debug remotely."""

    def __init__(
        self,
        stage: str,
        exc: BaseException,
        *,
        fallback_exc: BaseException | None = None,
    ):
        super().__init__(str(exc))
        self.stage = stage
        self.original = exc
        self.fallback_exc = fallback_exc

    def payload(self) -> dict[str, Any]:
        payload = {
            "error": "inference_failed",
            "stage": self.stage,
            "type": type(self.original).__name__,
            "message": str(self.original),
            "hint": _failure_hint(self.original),
        }
        if self.fallback_exc is not None:
            payload["fallback"] = {
                "type": type(self.fallback_exc).__name__,
                "message": str(self.fallback_exc),
                "hint": _failure_hint(self.fallback_exc),
            }
        return payload


def _failure_hint(exc: BaseException) -> str:
    message = f"{type(exc).__name__}: {exc}".lower()
    if any(x in message for x in ("gated", "401", "403", "llama", "access to model")):
        return (
            "The Hugging Face token cannot read meta-llama/Llama-3.2-3B. "
            "Accept that model's license or use TRIBEV2_TEXT_MODE=off/auto."
        )
    if any(x in message for x in ("uvx", "whisperx", "transcrib")):
        return (
            "WhisperX transcription failed. Install `uv`, or use the default "
            "auto mode so scoring retries with the configured ungated modalities."
        )
    if any(x in message for x in ("out of memory", "cuda", "cublas", "cudnn")):
        return (
            "GPU inference failed, commonly from T4 memory pressure. Auto mode "
            "retries without the LLaMA text encoder; shorten the clip if needed."
        )
    if any(x in message for x in ("ffmpeg", "moviepy", "decode", "moov atom")):
        return "The upload could not be decoded. Re-encode it as SDR H.264/AAC MP4."
    if any(x in message for x in ("xet", "huggingface", "download", "cdn.hf.co")):
        return (
            "A Hugging Face encoder download failed. Run the notebook's encoder "
            "prefetch cell; it bypasses hf-xet, verifies file size/SHA-256, and "
            "loads V-JEPA2 from a local directory."
        )
    return "Inspect /diagnostics and the server log for the exact failing stage."


def _release_transient_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _hf_token() -> str:
    return (
        os.getenv("HF_TOKEN")
        or os.getenv("HUGGING_FACE_HUB_TOKEN")
        or os.getenv("HUGGINGFACE_HUB_TOKEN")
        or ""
    ).strip()


def _resolve_text_capability() -> tuple[bool, str]:
    if TEXT_MODE == "off":
        return False, "disabled by TRIBEV2_TEXT_MODE=off"

    token = _hf_token()
    if not token:
        reason = (
            "HF_TOKEN is not set; using reliable "
            f"{'+'.join(INFERENCE_MODALITIES)} mode"
        )
        if TEXT_MODE == "required":
            reason += " (required mode will reject scoring)"
        return False, reason

    try:
        from huggingface_hub import hf_hub_download

        # A tiny gated file is a real authorization check. Merely querying model
        # metadata succeeds even when the account has not accepted the license.
        hf_hub_download(
            LLAMA_MODEL_ID,
            "config.json",
            token=token,
            cache_dir=str(CACHE_DIR / "huggingface"),
        )
    except Exception as exc:
        reason = (
            f"HF_TOKEN cannot access {LLAMA_MODEL_ID}: "
            f"{type(exc).__name__}: {exc}"
        )
        if TEXT_MODE == "required":
            reason += " (required mode will reject scoring)"
        return False, reason
    return True, f"authorized for {LLAMA_MODEL_ID}"


def _local_encoder_overrides() -> dict[str, str]:
    """Point TRIBE's extractors at verified local model snapshots when supplied."""
    overrides: dict[str, str] = {}
    local_paths = [path for path in (VIDEO_MODEL_PATH, AUDIO_MODEL_PATH) if path]
    if local_paths:
        from neuralset.extractors.base import HuggingFaceMixin

        for raw_path in local_paths:
            path = Path(raw_path)
            if not path.is_dir():
                raise FileNotFoundError(f"local encoder directory does not exist: {path}")
            if not (path / "config.json").is_file():
                raise FileNotFoundError(f"local encoder config is missing: {path / 'config.json'}")
            if not (path / "model.safetensors").is_file():
                raise FileNotFoundError(
                    f"local encoder weights are missing: {path / 'model.safetensors'}"
                )
            value = str(path)
            if value not in HuggingFaceMixin._REPOS:
                HuggingFaceMixin._REPOS.append(value)

    if VIDEO_MODEL_PATH:
        overrides["data.video_feature.image.model_name"] = VIDEO_MODEL_PATH
    if AUDIO_MODEL_PATH:
        overrides["data.audio_feature.model_name"] = AUDIO_MODEL_PATH
    return overrides


@asynccontextmanager
async def lifespan(_: FastAPI):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    _state["text_enabled"], _state["text_reason"] = _resolve_text_capability()

    try:
        from tribev2 import TribeModel

        if not torch.cuda.is_available():
            print(
                "[neuro] WARNING: CUDA unavailable; inference will be very slow.",
                flush=True,
            )
        print(
            f"[neuro] loading {MODEL_ID}; text={_state['text_enabled']} "
            f"({_state['text_reason']})",
            flush=True,
        )
        encoder_overrides = _local_encoder_overrides()
        model = TribeModel.from_pretrained(
            MODEL_ID,
            cache_folder=str(CACHE_DIR),
            config_update=encoder_overrides or None,
        )
        scorer = EngagementScorer(
            cache_dir=CACHE_DIR,
            tr_seconds=float(model.data.TR),
        )
        try:
            scorer.warmup()
        except Exception as exc:
            print(f"[neuro] atlas warmup deferred: {exc}", flush=True)
        _state["model"], _state["scorer"] = model, scorer
        print("[neuro] ready.", flush=True)
    except Exception as exc:
        _state["startup_error"] = {
            "type": type(exc).__name__,
            "message": str(exc),
            "traceback": traceback.format_exc(),
        }
        print(f"[neuro] startup failed:\n{traceback.format_exc()}", flush=True)

    yield
    _state["model"] = _state["scorer"] = None
    _release_transient_memory()


app = FastAPI(title="Cerebra neuro-engagement scorer", lifespan=lifespan)
app.add_middleware(
    CORSMiddleware,
    allow_origins=os.getenv("CORS_ORIGINS", "*").split(","),
    allow_methods=["POST", "GET"],
    allow_headers=["*"],
)


def _check_key(provided: str | None) -> None:
    if API_KEY and (provided or "").strip() != API_KEY:
        raise HTTPException(status_code=401, detail="invalid or missing x-api-key")


def _runtime_info() -> dict[str, Any]:
    cuda = torch.cuda.is_available()
    return {
        "ready": _state["model"] is not None,
        "server_version": SERVER_VERSION,
        "model": MODEL_ID,
        "upstream_commit": UPSTREAM_COMMIT,
        "cuda": cuda,
        "device": torch.cuda.get_device_name(0) if cuda else "cpu",
        "torch": torch.__version__,
        "auth": bool(API_KEY),
        "text_mode": TEXT_MODE,
        "text_enabled": _state["text_enabled"],
        "text_reason": _state["text_reason"],
        "inference_modalities": list(INFERENCE_MODALITIES),
        "video_model_path": VIDEO_MODEL_PATH or "huggingface",
        "audio_model_path": AUDIO_MODEL_PATH or "huggingface",
        "max_clip_seconds": MAX_CLIP_SECONDS,
        "max_loop_iterations": MAX_LOOP_ITERATIONS,
        "ad_score_feature_weights": FEATURE_WEIGHTS,
        "yolo_model": YOLO_MODEL,
        "startup_error": _state["startup_error"],
    }


@app.get("/health")
def health():
    return _runtime_info()


@app.get("/diagnostics")
def diagnostics(x_api_key: str | None = Header(default=None)):
    _check_key(x_api_key)
    return {**_runtime_info(), "last_error": _state["last_error"]}


def _base_video_events(video_path: str):
    import pandas as pd
    from neuralset.events.utils import standardize_events

    return standardize_events(
        pd.DataFrame(
            [
                {
                    "type": "Video",
                    "filepath": video_path,
                    "start": 0,
                    "timeline": "default",
                    "subject": "default",
                }
            ]
        )
    )


def _build_events(
    video_path: str,
    *,
    include_text: bool,
    include_audio: bool | None = None,
):
    """Build events without the upstream spaCy-heavy AddText round trip.

    WhisperX already supplies sentence and sequence information. A cumulative
    context is sufficient for TRIBE's contextualized Word extractor and avoids
    downloading ``en_core_web_lg`` during the first request.
    """

    from neuralset.events.transforms import (
        AddConcatenationContext,
        ChunkEvents,
        ExtractAudioFromVideo,
        RemoveMissing,
    )
    from neuralset.events.utils import standardize_events

    events = _base_video_events(video_path)
    warnings: list[str] = []
    if include_audio is None:
        include_audio = "audio" in INFERENCE_MODALITIES or include_text
    if include_audio:
        try:
            events = ExtractAudioFromVideo()(events)
        except Exception as exc:
            warnings.append(f"audio extraction skipped: {type(exc).__name__}: {exc}")

    for event_type in ("Audio", "Video"):
        try:
            events = ChunkEvents(
                event_type_to_chunk=event_type,
                max_duration=60,
                min_duration=30,
            )(events)
        except Exception as exc:
            warnings.append(
                f"{event_type.lower()} chunking skipped: "
                f"{type(exc).__name__}: {exc}"
            )

    if include_text and "Audio" in set(events["type"]):
        from tribev2.eventstransforms import ExtractWordsFromAudio

        events = ExtractWordsFromAudio()(events)
        if "Word" in set(events["type"]):
            events = AddConcatenationContext(
                sentence_only=False,
                max_context_len=1024,
                split_field="",
            )(events)
            events = RemoveMissing()(events)

    return standardize_events(events), warnings


def _drop_text_events(events):
    if "type" not in events.columns:
        return events
    return events.loc[
        ~events["type"].isin({"Word", "Text", "Sentence"})
    ].reset_index(drop=True)


def _predict_video(model: Any, video_path: str) -> tuple[np.ndarray, dict[str, Any]]:
    if TEXT_MODE == "required" and not _state["text_enabled"]:
        raise PipelineFailure(
            "text_preflight",
            RuntimeError(_state["text_reason"]),
        )

    use_text = bool(_state["text_enabled"] and TEXT_MODE != "off")
    warnings: list[str] = []
    try:
        events, event_warnings = _build_events(
            video_path,
            include_text=use_text,
            include_audio="audio" in INFERENCE_MODALITIES or use_text,
        )
        warnings.extend(event_warnings)
    except Exception as exc:
        if TEXT_MODE == "required":
            raise PipelineFailure("event_preprocessing", exc) from exc
        warnings.append(
            f"text preprocessing failed; retrying "
            f"{'+'.join(INFERENCE_MODALITIES)}: "
            f"{type(exc).__name__}: {exc}"
        )
        _release_transient_memory()
        try:
            events, event_warnings = _build_events(
                video_path,
                include_text=False,
                include_audio="audio" in INFERENCE_MODALITIES,
            )
            warnings.extend(event_warnings)
            use_text = False
        except Exception as fallback_exc:
            raise PipelineFailure(
                "event_preprocessing",
                exc,
                fallback_exc=fallback_exc,
            ) from fallback_exc

    try:
        predictions, _ = model.predict(events, verbose=False)
    except Exception as exc:
        has_text_events = (
            "type" in events.columns
            and events["type"].isin({"Word", "Text", "Sentence"}).any()
        )
        if TEXT_MODE != "required" and has_text_events:
            warnings.append(
                f"text inference failed; retrying "
                f"{'+'.join(INFERENCE_MODALITIES)}: "
                f"{type(exc).__name__}: {exc}"
            )
            _release_transient_memory()
            fallback_events = _drop_text_events(events)
            try:
                predictions, _ = model.predict(fallback_events, verbose=False)
                events = fallback_events
                use_text = False
            except Exception as fallback_exc:
                raise PipelineFailure(
                    "model_predict",
                    exc,
                    fallback_exc=fallback_exc,
                ) from fallback_exc
        else:
            raise PipelineFailure("model_predict", exc) from exc

    predictions = np.asarray(predictions)
    modalities = sorted(
        {
            {"Video": "video", "Audio": "audio", "Word": "text"}.get(t, "")
            for t in set(events["type"])
        }
        - {""}
    )
    return predictions, {
        "modalities": modalities,
        "text_used": bool(use_text and "text" in modalities),
        "warnings": warnings,
    }


def _run_locked(model: Any, video_path: str):
    with _inference_lock:
        return _predict_video(model, video_path)


def _reward_feedback(
    report: dict[str, Any],
    visual_features: dict[str, Any],
    yolo_features: dict[str, Any],
    ad_score: dict[str, Any],
) -> dict[str, Any]:
    """Turn the weakest weighted ad features into generator directions."""
    regions = report.get("regions", [])
    weakest = min(regions, key=lambda item: item["score"]) if regions else None
    weak_window = report.get("weakWindow", {})
    actions: list[str] = []
    weak_names = {item["name"] for item in ad_score["weakestFeatures"]}

    if weakest and any(name.startswith("tribe_") for name in weak_names):
        lever = {
            "LANG": "make the spoken/on-screen message clearer and more immediate",
            "VIS": "add a readable visual reveal or purposeful camera/subject motion",
            "AUD": "increase audio contrast with a beat, emphasis, or cleaner speech",
            "ATTN": "introduce a salient pattern interrupt or surprising change",
        }.get(weakest["short"], "strengthen the weakest cortical channel")
        actions.append(
            f"At {weak_window.get('startTime', 0)}–"
            f"{weak_window.get('endTime', 0)}s, {lever}."
        )

    sharpness = float(visual_features.get("sharpness", 500.0))
    brightness = float(visual_features.get("brightness", 0.5))
    motion = float(visual_features.get("motion_energy", 0.08))
    scene_rate = float(visual_features.get("scene_change_rate", 0.0))

    if "opencv_sharpness" in weak_names or sharpness < 180:
        actions.append("Increase subject sharpness and reduce blur/compression.")
    if "opencv_exposure" in weak_names and brightness < 0.5:
        actions.append("Raise exposure so the focal subject reads instantly.")
    elif "opencv_exposure" in weak_names and brightness >= 0.5:
        actions.append("Reduce clipped highlights and restore visible detail.")
    if "opencv_motion" in weak_names and motion < 0.08:
        actions.append("Add one controlled motion beat; the clip is visually static.")
    elif "opencv_motion" in weak_names and motion >= 0.08:
        actions.append("Reduce chaotic motion; preserve one clear focal action.")
    if "opencv_pacing" in weak_names or scene_rate > 0.5:
        actions.append("Use fewer cuts inside the three-second window.")
    if yolo_features.get("available"):
        if "yolo_detection_coverage" in weak_names:
            actions.append("Keep a recognizable product/person visible across more frames.")
        if "yolo_focal_size" in weak_names:
            actions.append("Make the primary product or subject occupy more of the frame.")
        if "yolo_centering" in weak_names:
            actions.append("Move the primary product or subject closer to the visual focal area.")
        if "yolo_focal_confidence" in weak_names:
            actions.append("Simplify the composition so the main subject is unambiguous.")
    if not actions:
        actions.append("Preserve the structure and make one small targeted variation.")

    return {
        "reward_metric": "adScore",
        "ad_score": ad_score["adScore"],
        "weakest_ad_features": ad_score["weakestFeatures"],
        "weakest_region": weakest,
        "weak_window": weak_window,
        "next_actions": actions[:3],
        "generator_instruction": " ".join(actions[:3]),
    }


async def _score_upload(
    video: UploadFile,
    model: Any,
    scorer: EngagementScorer,
) -> dict[str, Any]:
    suffix = Path(video.filename or "clip.mp4").suffix.lower() or ".mp4"
    if suffix not in {".mp4", ".avi", ".mkv", ".mov", ".webm"}:
        raise HTTPException(status_code=415, detail=f"unsupported video type: {suffix}")

    raw_path = normalized_path = None
    size = 0
    try:
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as raw:
            raw_path = raw.name
            while chunk := await video.read(1024 * 1024):
                size += len(chunk)
                if size > MAX_UPLOAD_BYTES:
                    raise HTTPException(
                        status_code=413,
                        detail=f"video exceeds {MAX_UPLOAD_BYTES} bytes",
                    )
                raw.write(chunk)
        if size == 0:
            raise HTTPException(status_code=400, detail="empty upload")

        with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as normalized:
            normalized_path = normalized.name
        try:
            visual_features = await run_in_threadpool(
                normalize_short_video,
                raw_path,
                normalized_path,
                max_seconds=MAX_CLIP_SECONDS,
            )
        except Exception as exc:
            failure = PipelineFailure("opencv_preprocessing", exc)
            payload = failure.payload()
            _state["last_error"] = {
                **payload,
                "traceback": traceback.format_exc(),
            }
            raise HTTPException(status_code=422, detail=payload) from exc

        yolo_features = await run_in_threadpool(
            extract_yolo_features,
            normalized_path,
            model_path=YOLO_MODEL,
        )
        try:
            predictions, inference = await run_in_threadpool(
                _run_locked,
                model,
                normalized_path,
            )
            report = scorer.score(predictions)
        except PipelineFailure as exc:
            payload = exc.payload()
            _state["last_error"] = {
                **payload,
                "traceback": traceback.format_exc(),
            }
            print(f"[neuro] request failed:\n{traceback.format_exc()}", flush=True)
            raise HTTPException(status_code=500, detail=payload) from exc
        except Exception as exc:
            failure = PipelineFailure("engagement_scoring", exc)
            payload = failure.payload()
            _state["last_error"] = {
                **payload,
                "traceback": traceback.format_exc(),
            }
            print(f"[neuro] scoring failed:\n{traceback.format_exc()}", flush=True)
            raise HTTPException(status_code=500, detail=payload) from exc

        ad_score = build_ad_score(report, visual_features, yolo_features)
        report["adScore"] = ad_score["adScore"]
        # Backward-compatible alias for existing clients/skills.
        report["engagementScore"] = ad_score["adScore"]
        report["adScoreBreakdown"] = ad_score
        report["video_name"] = video.filename
        report["inference"] = inference
        report["videoFeatures"] = visual_features
        report["yoloFeatures"] = yolo_features
        report["rewardFeedback"] = _reward_feedback(
            report,
            visual_features,
            yolo_features,
            ad_score,
        )
        _state["last_error"] = None
        return report
    finally:
        await video.close()
        for path in (raw_path, normalized_path):
            if path:
                Path(path).unlink(missing_ok=True)


@app.post("/score")
async def score(
    video: UploadFile = File(...),
    x_api_key: str | None = Header(default=None),
):
    _check_key(x_api_key)
    model, scorer = _state["model"], _state["scorer"]
    if model is None or scorer is None:
        raise HTTPException(
            status_code=503,
            detail={
                "error": "model_not_ready",
                "startup_error": _state["startup_error"],
            },
        )

    return await _score_upload(video, model, scorer)


@app.post("/train-loop")
async def train_loop(
    videos: list[UploadFile] = File(...),
    max_iterations: int = Form(default=MAX_LOOP_ITERATIONS),
    epsilon: float = Form(default=0.5),
    x_api_key: str | None = Header(default=None),
):
    """Score up to five <=3s candidates and keep the best non-regressing result.

    This is the evaluation/reward side of the optimization loop. Pika can
    generate one candidate per iteration, then submit the candidates here in
    order. The endpoint never accepts more than five iterations.
    """
    _check_key(x_api_key)
    model, scorer = _state["model"], _state["scorer"]
    if model is None or scorer is None:
        raise HTTPException(status_code=503, detail="model not ready")
    if not 1 <= max_iterations <= MAX_LOOP_ITERATIONS:
        raise HTTPException(
            status_code=422,
            detail=f"max_iterations must be between 1 and {MAX_LOOP_ITERATIONS}",
        )
    if not videos:
        raise HTTPException(status_code=422, detail="at least one candidate is required")
    if len(videos) > max_iterations:
        raise HTTPException(
            status_code=422,
            detail=(
                f"received {len(videos)} candidates but max_iterations="
                f"{max_iterations}"
            ),
        )

    history = []
    best_iteration = None
    best_score = float("-inf")
    no_improvement = 0
    stop_reason = "candidate_budget_exhausted"

    for index, video in enumerate(videos, start=1):
        report = await _score_upload(video, model, scorer)
        score_value = float(report["adScore"])
        prior_best = None if best_iteration is None else best_score
        reward = 0.0 if prior_best is None else round(score_value - prior_best, 1)
        accepted = best_iteration is None or score_value > best_score
        if accepted:
            best_score = score_value
            best_iteration = index
        if prior_best is not None and reward < epsilon:
            no_improvement += 1
        else:
            no_improvement = 0
        history.append(
            {
                "iteration": index,
                "candidate": report["video_name"],
                "score": score_value,
                "ad_score": score_value,
                "reward": reward,
                "best_score_before": prior_best,
                "accepted": accepted,
                "activation_score": report["activationScore"],
                "visual_score": report["videoFeatures"]["visual_score"],
                "weak_window": report["weakWindow"],
                "trimmed_to_seconds": report["videoFeatures"]["duration"],
                "next_action": report["rewardFeedback"]["generator_instruction"],
                "report": report,
            }
        )
        if no_improvement >= 2:
            stop_reason = f"plateau: reward < {epsilon} for two iterations"
            break

    return {
        "status": "completed",
        "iterations_run": len(history),
        "max_iterations": max_iterations,
        "max_video_seconds": MAX_CLIP_SECONDS,
        "best_iteration": best_iteration,
        "best_score": round(best_score, 1),
        "stop_reason": stop_reason,
        "history": history,
    }


## 4. Launch + tunnel
This starts a logged API process, verifies model readiness, then creates the public tunnel. The cell may finish; the subprocesses stay alive for the Colab runtime. Wait for `READY` before scoring.

In [ ]:
# 4. Launch the scorer + a public tunnel. Copy the printed https URL into the
#    Pika skill (NEURO_API_URL). Set a key so the public tunnel isn't open.
import hashlib, os, re, sys, time, subprocess, secrets, urllib.request, json
from pathlib import Path

# ---- config -------------------------------------------------------------
os.environ["TRIBEV2_CACHE_DIR"] = "/content/tribev2-cache"
# Universal default: no gated LLaMA access is required. Change this to "auto"
# only if you have accepted the LLaMA 3.2 license and configured HF_TOKEN.
os.environ["TRIBEV2_TEXT_MODE"] = "off"
os.environ["TRIBEV2_MODALITIES"] = "video"
os.environ["TRIBEV2_UPSTREAM_COMMIT"] = "38b3db073a7fa5bfbfc7fdd894b1a3536e4553e3"
# hf-xet stalled during a real Colab T4 test after transferring ~3 GB of the
# 4.14 GB V-JEPA2 checkpoint, leaving a 768 MB incomplete file and a request
# hanging for 30 minutes. The verified encoder prefetch below uses curl instead.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["YOLO_MODEL"] = "/content/yolov8n.pt"
os.environ.setdefault("NEURO_API_KEY", secrets.token_urlsafe(16))  # shared secret
from google.colab import userdata
try:
    _hf_token = userdata.get("HF_TOKEN")
    if _hf_token:
        os.environ["HF_TOKEN"] = _hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf_token
        print("HF_TOKEN found. The server will verify LLaMA 3.2 access.")
    else:
        print("HF_TOKEN is empty — public video-only mode still works.")
except Exception:
    print("No readable HF_TOKEN secret — public video-only mode still works.")

# ---- deterministic encoder prefetch --------------------------------------
# Do not let the first user's /score request become a hidden 4 GB model
# download. Fetch to a normal local directory, resume safely, enforce a
# minimum transfer speed, and verify both exact size and SHA-256.
ENCODER_ROOT = Path("/content/tribev2-encoders")
VIDEO_ENCODER = ENCODER_ROOT / "vjepa2-vitg-fpc64-256"

def fetch_public_model(repo, destination, expected_size, expected_sha):
    destination.mkdir(parents=True, exist_ok=True)
    files = ("config.json", "video_preprocessor_config.json", "model.safetensors")

    def file_sha256(path):
        digest = hashlib.sha256()
        with open(path, "rb") as stream:
            while chunk := stream.read(16 * 1024 * 1024):
                digest.update(chunk)
        return digest.hexdigest()

    for filename in files:
        target = destination / filename
        url = f"https://huggingface.co/{repo}/resolve/main/{filename}"
        if filename != "model.safetensors":
            subprocess.run(
                ["curl", "-L", "--fail", "--retry", "5", "--retry-all-errors",
                 "--connect-timeout", "30", "-o", str(target), url],
                check=True,
            )
            continue

        if target.exists():
            size_ok = target.stat().st_size == expected_size
            sha_ok = size_ok and file_sha256(target) == expected_sha
            if size_ok and sha_ok:
                print(f"{repo} already verified ({expected_size / 1e9:.2f} GB).")
                continue
            target.unlink()
        partial = target.with_suffix(target.suffix + ".part")
        if partial.exists() and partial.stat().st_size > expected_size:
            partial.unlink()
        command = [
            "curl", "-L", "--fail", "--retry", "12", "--retry-all-errors",
            "--connect-timeout", "30", "--speed-limit", "1024",
            "--speed-time", "120", "-o", str(partial),
        ]
        if partial.exists():
            command += ["-C", "-"]
        command.append(url)
        print(f"Downloading {repo} ({expected_size / 1e9:.2f} GB); resumable...")
        subprocess.run(command, check=True)
        actual_size = partial.stat().st_size
        if actual_size != expected_size:
            raise RuntimeError(
                f"{repo} size mismatch: expected {expected_size}, got {actual_size}"
            )
        actual_sha = file_sha256(partial)
        if actual_sha != expected_sha:
            raise RuntimeError(
                f"{repo} SHA-256 mismatch: expected {expected_sha}, got {actual_sha}"
            )
        partial.replace(target)
        print(f"Verified {repo}: {actual_size / 1e9:.2f} GB, SHA-256 OK.")

fetch_public_model(
    "facebook/vjepa2-vitg-fpc64-256",
    VIDEO_ENCODER,
    4_138_311_608,
    "f205e77aa2ade168db6b09d4bc420d156141f64ab964278a9c181a2bdf2a232b",
)
os.environ["TRIBEV2_VIDEO_MODEL_PATH"] = str(VIDEO_ENCODER)
print("TRIBE video encoder ready:", VIDEO_ENCODER)

# Public 6 MB YOLO-nano weights; ungated, integrity-checked, and cached.
_yolo_path = Path(os.environ["YOLO_MODEL"])
_expected_yolo_sha = "f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36"
if not _yolo_path.exists() or hashlib.sha256(_yolo_path.read_bytes()).hexdigest() != _expected_yolo_sha:
    urllib.request.urlretrieve("https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt", _yolo_path)
_actual_yolo_sha = hashlib.sha256(_yolo_path.read_bytes()).hexdigest()
assert _actual_yolo_sha == _expected_yolo_sha, (
    f"YOLO weight checksum mismatch: {_actual_yolo_sha}"
)
from ultralytics import YOLO
YOLO(os.environ["YOLO_MODEL"])
print("YOLO ready:", os.environ["YOLO_MODEL"])

# ---- start the API and keep a durable log --------------------------------
# A notebook rerun should replace only its own stale scorer/tunnel processes.
subprocess.run(["pkill", "-f", "uvicorn server:app"], check=False)
subprocess.run(["pkill", "-f", "/content/cloudflared tunnel"], check=False)
time.sleep(2)
LOG_PATH = Path("/content/neuro-server.log")
api_log = open(LOG_PATH, "w", buffering=1)
api = subprocess.Popen([sys.executable, "-m", "uvicorn", "server:app",
                        "--host", "0.0.0.0", "--port", "8000"],
                       cwd="/content", stdout=api_log, stderr=subprocess.STDOUT,
                       text=True)

def show_log_tail(lines=80):
    api_log.flush()
    if LOG_PATH.exists():
        print("\n--- neuro-server.log (tail) ---")
        print("\n".join(LOG_PATH.read_text(errors="replace").splitlines()[-lines:]))
        print("--- end log ---\n")

print("Loading TRIBE v2 (checkpoint is ~676 MB)...")
health = None
for _ in range(180):  # up to 30 minutes for a cold Colab cache
    if api.poll() is not None:
        show_log_tail()
        raise RuntimeError(f"API process exited with code {api.returncode}")
    try:
        health = json.load(urllib.request.urlopen("http://localhost:8000/health", timeout=5))
        if health.get("startup_error"):
            show_log_tail()
            raise RuntimeError("TRIBE startup failed: " + json.dumps(health["startup_error"], indent=2))
        if health.get("ready"):
            break
    except RuntimeError:
        raise
    except Exception:
        pass
    time.sleep(10)
else:
    show_log_tail()
    raise TimeoutError("TRIBE did not become ready within 30 minutes")

print("READY:", json.dumps(health, indent=2))

# ---- public tunnel via cloudflared (no account needed) -------------------
if not Path("/content/cloudflared").exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000",
                        "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for line in tun.stdout:
    print(line, end="")
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0); break

print("\n" + "=" * 64)
print("NEURO_API_URL =", public_url)
print("NEURO_API_KEY =", os.environ["NEURO_API_KEY"])
print("=" * 64)
print("Paste both into the Pika neuro-eval / neuro-optimize skill.")
print("Text mode:", health["text_reason"])
print("Server log:", LOG_PATH)


## 5. Score a clip → activation score
Runs against `localhost:8000` (no tunnel timeout). Upload a video (or set `VIDEO` to a URL/path). It is normalized to at most 3 seconds, then scored using a transparent linear combination of TRIBE, OpenCV, and YOLO features. On failure this prints the structured stage, diagnostics, and server-log tail instead of hiding everything behind `HTTPError: 500`.

In [ ]:
# 5. Extract the activation / engagement score for a clip.
#    Hits the LOCAL server (localhost:8000), so the cloudflared 100s edge
#    timeout never applies — a long first call still returns here.
import os, json, requests
from pathlib import Path

# Set VIDEO to a URL/path, or leave it as None to upload from your computer.
VIDEO = None
if VIDEO is None:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No video uploaded")
    filename, content = next(iter(uploaded.items()))
    path = f"/content/{Path(filename).name}"
    Path(path).write_bytes(content)
elif VIDEO.startswith("http"):
    path = "/content/clip.mp4"
    download = requests.get(VIDEO, timeout=120)
    download.raise_for_status()
    Path(path).write_bytes(download.content)
else:
    path = VIDEO

print("Scoring on the T4 (first inference is slowest — encoder warm-up)…")
with open(path, "rb") as clip:
    r = requests.post("http://localhost:8000/score",
                      headers={"x-api-key": os.environ["NEURO_API_KEY"]},
                      files={"video": (Path(path).name, clip, "video/mp4")},
                      timeout=1800)
if not r.ok:
    try:
        error_body = r.json()
    except Exception:
        error_body = r.text
    print("SCORING FAILED:", r.status_code, json.dumps(error_body, indent=2))
    try:
        d = requests.get("http://localhost:8000/diagnostics",
                         headers={"x-api-key": os.environ["NEURO_API_KEY"]},
                         timeout=30).json()
        print("DIAGNOSTICS:", json.dumps(d, indent=2))
    except Exception as diag_exc:
        print("Could not fetch diagnostics:", diag_exc)
    log_path = Path("/content/neuro-server.log")
    if log_path.exists():
        print("\nSERVER LOG (last 120 lines):")
        print("\n".join(log_path.read_text(errors="replace").splitlines()[-120:]))
    raise RuntimeError(f"Scoring failed at HTTP {r.status_code}; details printed above")
rep = r.json()

peak, weak = rep.get("peak", {}), rep.get("weakWindow", {})
print("\n" + "=" * 62)
print(f"AD SCORE: {rep['adScore']}/100   ({rep['duration']}s)")
print("-" * 62)
for x in rep["regions"]:                      # per-region cortical activation
    print(f"  {x['short']:5} {x['name']:26} {x['score']:5.1f}   ({x['reliability']})")
print("-" * 62)
print(f"  Peak  {peak.get('time')}s  ({peak.get('label')})  {peak.get('value')}")
print(f"  Weak  {weak.get('startTime')}–{weak.get('endTime')}s  mean {weak.get('meanValue')}  <- regenerate this beat")
print(f"  Modalities: {', '.join(rep['inference']['modalities'])}"
      f" | text_used={rep['inference']['text_used']}")
print("  Feature contributions:")
for feature in rep["adScoreBreakdown"]["features"]:
    score = feature["score"]
    if score is not None:
        print(f"    {feature['name']:27} score={score:5.1f} "
              f"weight={feature['effective_weight']:.3f} "
              f"contribution={feature['contribution']:.2f}")
print("  Next:", rep["rewardFeedback"]["generator_instruction"])
for warning in rep["inference"].get("warnings", []):
    print("  Warning:", warning)
print("=" * 62)
# rep['global'] = per-frame engagement curve; rep['cognitiveSeries'] = per-family traces.


## 6. Run up to five optimization iterations
Upload 1–5 candidate clips. Each is trimmed to ≤3 seconds; the loop keeps the best score, never regresses, and stops on a two-iteration plateau.

In [ ]:
# 6. Bounded optimization/training loop (maximum 5 candidates, 3 seconds each).
#
# Upload candidates in iteration order. The API normalizes every upload to <=3s,
# scores it with TRIBE activations + OpenCV features, keeps only improvements,
# and stops after two sub-epsilon gains. This is a reward/selection loop; Pika's
# neuro-optimize skill can generate each next candidate from the prior diagnosis.
import json, os, requests
from google.colab import files

MAX_ITERATIONS = 5
EPSILON = 0.5
print(f"Upload 1–{MAX_ITERATIONS} candidate videos in iteration order.")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No candidates uploaded")
if len(uploaded) > MAX_ITERATIONS:
    raise ValueError(f"Upload at most {MAX_ITERATIONS} candidates")

multipart = [
    ("videos", (name, content, "video/mp4"))
    for name, content in uploaded.items()
]
r = requests.post(
    "http://localhost:8000/train-loop",
    headers={"x-api-key": os.environ["NEURO_API_KEY"]},
    data={"max_iterations": MAX_ITERATIONS, "epsilon": EPSILON},
    files=multipart,
    timeout=1800,
)
if not r.ok:
    print(json.dumps(r.json(), indent=2))
    r.raise_for_status()
loop = r.json()
print("\nBEST ITERATION:", loop["best_iteration"], "SCORE:", loop["best_score"])
print("STOP:", loop["stop_reason"])
for item in loop["history"]:
    marker = "✓" if item["accepted"] else "×"
    print(
        f"{marker} iter {item['iteration']}: {item['candidate']} "
        f"score={item['score']} activation={item['activation_score']} "
        f"visual={item['visual_score']} reward={item['reward']}"
    )
    print("   next:", item["next_action"])
